# Open QA Eval: FT checkpoint vs OpenAI API, Gemini judge

Notebook này dùng bộ `open_qa_eval.jsonl` có context tốt hơn để đánh giá:
- Model FT từ checkpoint đã lưu: `trainer/checkpoint-117`
- OpenAI API model để làm baseline so sánh
- Gemini round-robin để chấm từng câu theo cùng rubric

Metric chính:
- Độ đúng so với answer label/reference
- Độ bám context
- Độ phủ ý chính
- Có hallucination hay không
- Clarity
- Format một đoạn
- Composite score và win-rate FT vs OpenAI

Ghi chú runtime: Qwen3.5 cần `transformers>=5.2.0`; cell 0 đã bỏ `sentence-transformers` để tránh downgrade Transformers về 4.x.


In [ ]:
# 0) Install dependencies for Qwen3.5 + Unsloth
# IMPORTANT:
# - Qwen3.5 needs Transformers >= 5.2.0.
# - Current Unsloth requires Transformers <= 5.5.0.
# - So DO NOT use unconstrained "transformers>=5.2.0"; it may install 5.8.0 and break Unsloth.
# - After this cell finishes, runtime is intentionally restarted. Then run from cell 0.5 / cell 1.

!pip uninstall -y sentence-transformers torchcodec -q
!pip install -q --upgrade "unsloth" "unsloth_zoo" "peft" "accelerate" "bitsandbytes" "google-genai>=1.66.0,<2.0.0" "openai" "gdown" "tqdm" "pandas==2.2.2" "pillow<12"
!pip install -q --upgrade --force-reinstall --no-deps "transformers==5.2.0"
!pip install -q --upgrade --force-reinstall --no-deps "google-auth==2.47.0"

import os, time
print("Install done. Restarting runtime now. After restart, run from cell 0.5 then cell 1 onward.")
time.sleep(2)
os.kill(os.getpid(), 9)


In [ ]:
# 0.5) Version check after runtime restart
# Run this after cell 0 restarts the runtime, before loading the FT model.

import transformers
from packaging import version

tf_ver = version.parse(transformers.__version__)
print("Transformers version:", transformers.__version__)

if tf_ver < version.parse("5.2.0") or tf_ver > version.parse("5.5.0"):
    raise RuntimeError(
        f"Transformers {transformers.__version__} is not in the safe range for this setup. "
        "Need 5.2.0 <= transformers <= 5.5.0. "
        "Run cell 0 again. If pip still picks the wrong version, run: "
        '!pip install -q --upgrade --force-reinstall --no-deps "transformers==5.2.0"'
    )

print("Transformers version is OK for Qwen3.5 + current Unsloth.")


In [ ]:
# 1) Config + mount Drive
from getpass import getpass
from pathlib import Path
import json, time, re, os, math, random
from collections import defaultdict, Counter

import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

drive.mount("/content/drive")

# ===== Main switches =====
RUN_FT_PREDICT = True
RUN_OPENAI_PREDICT = True
RUN_GEMINI_JUDGE = True
RUN_SUMMARY = True
RUN_RANDOM_REVIEW = True

FORCE_RERUN_PREDICT = False
FORCE_REJUDGE = False

# None = full eval. Set 10 for smoke test.
MAX_EVAL_SAMPLES = None
EVAL_SEED = 42

# ===== Dataset =====
DATA_DRIVE_FILE_ID = "1dUPv3tUaGdelMsTFEQtKIBPtO3kK-arf"
DATA_FILENAME = "open_qa_eval.jsonl"

WORKDIR = Path("/content/drive/MyDrive/eduvidqa_qwen35_vl_full_r16e1")
OUTPUT_DIR = WORKDIR / "outputs"
DATA_CACHE_DIR = Path("/content/drive/MyDrive/eduvidqa_open_qa_eval")
DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DATA_JSONL_PATH = DATA_CACHE_DIR / DATA_FILENAME

# ===== FT checkpoint from training notebook =====
BASE_MODEL = "unsloth/Qwen3.5-4B"
FINAL_RUN_NAME = "final_fulltrain_h100_r16_e1_maxspeed"
FINAL_DIR = OUTPUT_DIR / "final_full_r16e1" / FINAL_RUN_NAME
CHECKPOINT_PATH = FINAL_DIR / "trainer" / "checkpoint-117"
FINAL_ADAPTER_PATH = FINAL_DIR / "adapter"

# ===== Generation config =====
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = False
GEN_MAX_NEW_TOKENS = 160
GEN_TEMPERATURE = 0.0
GEN_TOP_P = 1.0

# ===== OpenAI prediction config =====
OPENAI_PREDICT_MODEL = "gpt-4o-mini"
OPENAI_MAX_OUTPUT_TOKENS = 180
OPENAI_RPM_LIMIT = 60
OPENAI_API_KEY = getpass("OpenAI API key for prediction: ").strip()
assert OPENAI_API_KEY, "Need OpenAI API key for OpenAI prediction baseline."

# ===== Gemini judge config =====
GEMINI_MODEL = "gemini-3.1-flash-lite-preview"
GEMINI_THINKING_LEVEL = "HIGH"
GEMINI_MAX_OUTPUT_TOKENS = 4096
GEMINI_RPM_LIMIT_PER_KEY = 15
GEMINI_RPD_LIMIT_PER_KEY = 500
GEMINI_RATE_LIMIT_COOLDOWN_SEC = 60
MAX_RETRIES_PER_SAMPLE = 6

GEMINI_API_KEY_1 = getpass("Gemini API key #1: ").strip()
GEMINI_API_KEY_2 = getpass("Gemini API key #2: ").strip()
GEMINI_API_KEY_3 = getpass("Gemini API key #3: ").strip()
GEMINI_API_KEYS = [GEMINI_API_KEY_1, GEMINI_API_KEY_2, GEMINI_API_KEY_3]
assert all(GEMINI_API_KEYS), "Need 3 Gemini API keys."
assert len(set(GEMINI_API_KEYS)) == len(GEMINI_API_KEYS), "Three Gemini keys should be different."

# ===== Output paths =====
RUN_TAG = f"open_qa_ft_checkpoint117_vs_openai_{OPENAI_PREDICT_MODEL}_gemini_{GEMINI_THINKING_LEVEL.lower()}"
RUN_TAG = RUN_TAG.replace("/", "_").replace(":", "_")
EVAL_DIR = OUTPUT_DIR / "open_qa_eval" / RUN_TAG
EVAL_DIR.mkdir(parents=True, exist_ok=True)

FT_PRED_PATH = EVAL_DIR / "ft_checkpoint117_predictions.jsonl"
OPENAI_PRED_PATH = EVAL_DIR / "openai_predictions.jsonl"
JUDGE_PATH = EVAL_DIR / "gemini_pairwise_judge.jsonl"
SUMMARY_OUT = EVAL_DIR / "summary.json"
COMPARISON_CSV_OUT = EVAL_DIR / "comparison_metrics.csv"
PAIRWISE_CSV_OUT = EVAL_DIR / "pairwise_item_scores.csv"
RANDOM_REVIEW_JSONL = EVAL_DIR / "random_review_examples.jsonl"
RANDOM_REVIEW_CSV = EVAL_DIR / "random_review_examples.csv"
RANDOM_REVIEW_MD = EVAL_DIR / "random_review_examples.md"

print("Dataset path:", DATA_JSONL_PATH)
print("Checkpoint path:", CHECKPOINT_PATH)
print("Fallback adapter path:", FINAL_ADAPTER_PATH)
print("Eval output dir:", EVAL_DIR)

In [ ]:
# 2) Utility functions

def read_jsonl(path):
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def read_jsonl_safe(path):
    path = Path(path)
    rows, bad = [], []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                rows.append(json.loads(raw))
            except json.JSONDecodeError as exc:
                bad.append({"line_no": line_no, "error": str(exc), "prefix": raw[:300]})
    if bad:
        bad_path = path.with_suffix(path.suffix + ".bad_lines.json")
        write_json(bad_path, bad)
        print(f"[WARN] skipped {len(bad)} bad jsonl lines in {path}; saved -> {bad_path}")
    return rows

def append_jsonl(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def overwrite_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

def normalize_text(s):
    return re.sub(r"\s+", " ", str(s or "")).strip()

def get_ref_answer(row):
    ref = row.get("reference", {}) or {}
    parts = []
    if ref.get("answer_text"):
        parts.append(str(ref["answer_text"]))
    if ref.get("explanation"):
        parts.append(str(ref["explanation"]))
    return normalize_text(" ".join(parts))

def get_short_ref_answer(row):
    return normalize_text((row.get("reference", {}) or {}).get("answer_text", ""))

def get_question(row):
    return normalize_text((row.get("input", {}) or {}).get("question", ""))

def get_context(row, max_chars=6500):
    ctx = ((row.get("input", {}) or {}).get("context", {}) or {})
    pieces = []
    meta = row.get("metadata", {}) or {}
    if meta.get("lecture_title"):
        pieces.append(f"Lecture: {meta.get('lecture_title')}")
    if meta.get("unit_name"):
        pieces.append(f"Unit: {meta.get('unit_name')}")
    if ctx.get("unit_description"):
        pieces.append("Unit description: " + normalize_text(ctx.get("unit_description")))
    if ctx.get("unit_summary"):
        pieces.append("Unit summary: " + normalize_text(ctx.get("unit_summary")))
    kp = ctx.get("kp") or {}
    if kp:
        pieces.append("Knowledge point: " + normalize_text(f"{kp.get('name', '')}. {kp.get('description', '')}"))
    key_points = ctx.get("unit_key_points") or []
    if key_points:
        kp_lines = []
        for p in key_points[:10]:
            ts = p.get("timestamp_s")
            ts_txt = f"[ts={ts}s] " if ts is not None else ""
            kp_lines.append("- " + ts_txt + normalize_text(p.get("text", "")))
        pieces.append("Key points:\n" + "\n".join(kp_lines))
    if ctx.get("source_evidence_span"):
        pieces.append("Source evidence: " + normalize_text(ctx.get("source_evidence_span")))
    tw = ctx.get("transcript_window") or {}
    if tw.get("text"):
        pieces.append("Transcript window:\n" + str(tw.get("text"))[:3500])
    text = "\n\n".join(pieces)
    if len(text) > max_chars:
        text = text[:max_chars//2] + "\n...[TRUNCATED]...\n" + text[-max_chars//2:]
    return text

def build_answer_prompt(row):
    return f"""You are answering an educational question using only the provided lecture context.

Requirements:
- Answer in one concise paragraph.
- Do not use bullet points, markdown tables, JSON, or code fences.
- Do not reveal hidden reasoning.
- If the context is insufficient, say what can be answered from the context instead of inventing details.

Lecture context:
{get_context(row)}

Question:
{get_question(row)}

Answer:"""

def get_id(row):
    return row.get("eval_id") or row.get("id") or row.get("item_id")

def count_by(rows, key):
    return dict(Counter((r.get(key, "unknown") if isinstance(r, dict) else "unknown") for r in rows))

def prediction_answer(row):
    return normalize_text(row.get("prediction", ""))

def mean(vals):
    vals = [v for v in vals if v is not None]
    return sum(vals) / len(vals) if vals else None

In [ ]:
# 3) Load open_qa_eval.jsonl

if not DATA_JSONL_PATH.exists():
    print("Dataset file not found on Drive cache; downloading via gdown...")
    !gdown {DATA_DRIVE_FILE_ID} -O {DATA_JSONL_PATH}

assert DATA_JSONL_PATH.exists(), f"Dataset file not found: {DATA_JSONL_PATH}"

eval_rows = read_jsonl(DATA_JSONL_PATH)
print("Loaded rows:", len(eval_rows))

# Full schema validation over every row, not just the first few rows.
missing = Counter()
duplicate_ids = []
seen_ids = set()
context_word_counts = []
reference_word_counts = []
question_word_counts = []

for r in eval_rows:
    eid = get_id(r)
    if not eid:
        missing["eval_id/id/item_id"] += 1
    elif eid in seen_ids:
        duplicate_ids.append(eid)
    else:
        seen_ids.add(eid)

    if not get_question(r):
        missing["input.question"] += 1
    if not get_ref_answer(r):
        missing["reference.answer_text/explanation"] += 1
    if not get_context(r):
        missing["input.context"] += 1

    question_word_counts.append(len(get_question(r).split()))
    reference_word_counts.append(len(get_ref_answer(r).split()))
    context_word_counts.append(len(get_context(r).split()))

assert not missing, f"Schema missing fields: {dict(missing)}"
assert not duplicate_ids, f"Duplicate eval_id found, e.g. {duplicate_ids[:5]}"

if MAX_EVAL_SAMPLES is not None:
    rng = random.Random(EVAL_SEED)
    eval_rows = rng.sample(eval_rows, k=min(MAX_EVAL_SAMPLES, len(eval_rows)))

print("Eval rows used:", len(eval_rows))
print("Split counts:", count_by(eval_rows, "split"))
print("Difficulty counts:", count_by([r.get("metadata", {}) for r in eval_rows], "difficulty"))
print("Intent counts:", count_by([r.get("metadata", {}) for r in eval_rows], "question_intent"))
print("Lecture count:", len(set((r.get("metadata", {}) or {}).get("lecture_id") for r in eval_rows)))
print("Context words mean/max:", round(sum(context_word_counts)/len(context_word_counts), 1), max(context_word_counts))
print("Reference words mean/max:", round(sum(reference_word_counts)/len(reference_word_counts), 1), max(reference_word_counts))
print("Question words mean/max:", round(sum(question_word_counts)/len(question_word_counts), 1), max(question_word_counts))
print("First id:", get_id(eval_rows[0]))
print("First question:", get_question(eval_rows[0]))
print("First reference:", get_short_ref_answer(eval_rows[0]))


In [ ]:
# 4) FT checkpoint prediction

def load_ft_model():
    from unsloth import FastVisionModel
    from peft import PeftModel
    import torch

    active_adapter_path = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else FINAL_ADAPTER_PATH
    assert active_adapter_path.exists(), f"Adapter/checkpoint path not found: {active_adapter_path}"

    print("Loading base model:", BASE_MODEL)
    print("Loading adapter/checkpoint:", active_adapter_path)

    model, processor = FastVisionModel.from_pretrained(
        BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        use_gradient_checkpointing=False,
    )
    model = PeftModel.from_pretrained(model, str(active_adapter_path))
    FastVisionModel.for_inference(model)
    model.eval()
    return model, processor, active_adapter_path

def generate_ft_answer(model, processor, prompt):
    import torch

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], return_tensors="pt")

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items() if hasattr(v, "to")}

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=(GEN_TEMPERATURE > 0),
        top_p=GEN_TOP_P,
        use_cache=True,
    )
    if GEN_TEMPERATURE > 0:
        gen_kwargs["temperature"] = GEN_TEMPERATURE

    with torch.inference_mode():
        output_ids = model.generate(**gen_kwargs)

    input_len = inputs["input_ids"].shape[1]
    gen_ids = output_ids[:, input_len:]
    return normalize_text(processor.batch_decode(gen_ids, skip_special_tokens=True)[0])

def run_ft_predictions():
    if FT_PRED_PATH.exists() and not FORCE_RERUN_PREDICT:
        existing = read_jsonl_safe(FT_PRED_PATH)
        if len(existing) >= len(eval_rows):
            print("FT predictions already complete:", FT_PRED_PATH)
            return existing

    model, processor, active_adapter_path = load_ft_model()
    done_rows = [] if FORCE_RERUN_PREDICT else read_jsonl_safe(FT_PRED_PATH)
    done_ids = {r.get("eval_id") for r in done_rows}

    for row in tqdm(eval_rows, desc="FT predict", unit="sample"):
        eid = get_id(row)
        if eid in done_ids:
            continue

        prompt = build_answer_prompt(row)
        t0 = time.time()
        try:
            answer = generate_ft_answer(model, processor, prompt)
            status, err = "ok", None
        except Exception as exc:
            answer, status, err = "", "failed", str(exc)[:1000]
            print("[WARN] FT predict failed:", eid, err)

        out = {
            "eval_id": eid,
            "split": row.get("split"),
            "model": "ft_checkpoint117",
            "adapter_path": str(active_adapter_path),
            "question": get_question(row),
            "reference_answer": get_ref_answer(row),
            "prediction": answer,
            "latency_sec": time.time() - t0,
            "status": status,
            "error": err,
        }
        append_jsonl(FT_PRED_PATH, out)
        done_rows.append(out)
        done_ids.add(eid)

    return read_jsonl_safe(FT_PRED_PATH)

if RUN_FT_PREDICT:
    ft_predictions = run_ft_predictions()
else:
    ft_predictions = read_jsonl_safe(FT_PRED_PATH)

print("FT predictions:", len(ft_predictions), FT_PRED_PATH)

In [ ]:
# 5) OpenAI API prediction baseline

from openai import OpenAI
openai_client = OpenAI(api_key=OPENAI_API_KEY)

def openai_predict_one(prompt):
    system_msg = (
        "You answer educational questions using only the supplied lecture context. "
        "Return one concise paragraph. Do not use markdown, bullets, JSON, or code fences."
    )
    resp = openai_client.chat.completions.create(
        model=OPENAI_PREDICT_MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_tokens=OPENAI_MAX_OUTPUT_TOKENS,
    )
    usage = getattr(resp, "usage", None)
    return normalize_text(resp.choices[0].message.content or ""), {
        "prompt_tokens": int(getattr(usage, "prompt_tokens", 0) or 0) if usage else 0,
        "completion_tokens": int(getattr(usage, "completion_tokens", 0) or 0) if usage else 0,
        "total_tokens": int(getattr(usage, "total_tokens", 0) or 0) if usage else 0,
    }

def run_openai_predictions():
    if OPENAI_PRED_PATH.exists() and not FORCE_RERUN_PREDICT:
        existing = read_jsonl_safe(OPENAI_PRED_PATH)
        if len(existing) >= len(eval_rows):
            print("OpenAI predictions already complete:", OPENAI_PRED_PATH)
            return existing

    done_rows = [] if FORCE_RERUN_PREDICT else read_jsonl_safe(OPENAI_PRED_PATH)
    done_ids = {r.get("eval_id") for r in done_rows}
    last_call = 0.0

    for row in tqdm(eval_rows, desc="OpenAI predict", unit="sample"):
        eid = get_id(row)
        if eid in done_ids:
            continue

        min_gap = 60.0 / max(1, OPENAI_RPM_LIMIT)
        wait = max(0.0, min_gap - (time.monotonic() - last_call))
        if wait:
            time.sleep(wait)

        prompt = build_answer_prompt(row)
        t0 = time.time()
        try:
            answer, usage = openai_predict_one(prompt)
            status, err = "ok", None
        except Exception as exc:
            answer = ""
            usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
            status, err = "failed", str(exc)[:1000]
            print("[WARN] OpenAI predict failed:", eid, err)

        last_call = time.monotonic()
        out = {
            "eval_id": eid,
            "split": row.get("split"),
            "model": OPENAI_PREDICT_MODEL,
            "question": get_question(row),
            "reference_answer": get_ref_answer(row),
            "prediction": answer,
            "latency_sec": time.time() - t0,
            "status": status,
            "error": err,
            **usage,
        }
        append_jsonl(OPENAI_PRED_PATH, out)
        done_rows.append(out)
        done_ids.add(eid)

    return read_jsonl_safe(OPENAI_PRED_PATH)

if RUN_OPENAI_PREDICT:
    openai_predictions = run_openai_predictions()
else:
    openai_predictions = read_jsonl_safe(OPENAI_PRED_PATH)

print("OpenAI predictions:", len(openai_predictions), OPENAI_PRED_PATH)
print("OpenAI token usage:", {
    "prompt_tokens": sum(int(r.get("prompt_tokens", 0) or 0) for r in openai_predictions),
    "completion_tokens": sum(int(r.get("completion_tokens", 0) or 0) for r in openai_predictions),
    "total_tokens": sum(int(r.get("total_tokens", 0) or 0) for r in openai_predictions),
})

In [ ]:
# 6) Local rule metrics for predictions

def compute_rule_metrics(pred_rows):
    answers = [prediction_answer(r) for r in pred_rows if r.get("status", "ok") == "ok"]
    n = len(answers) or 1

    def has_bullets(a):
        return bool(re.search(r"(?m)^\s*([-*]|\d+[.)])\s+", a))

    def has_code_or_json(a):
        s = a.strip()
        return "```" in s or s.startswith("{") or s.startswith("[") or s.lower().startswith("json")

    def one_paragraph(a):
        return "\n" not in a.strip()

    def no_think(a):
        low = a.lower()
        return "<think>" not in low and "</think>" not in low

    def format_pass(a):
        return bool(a.strip()) and no_think(a) and one_paragraph(a) and not has_bullets(a) and not has_code_or_json(a)

    word_counts = [len(a.split()) for a in answers]

    return {
        "sample_count": len(pred_rows),
        "ok_count": sum(1 for r in pred_rows if r.get("status", "ok") == "ok"),
        "format_pass_rate": sum(format_pass(a) for a in answers) / n,
        "no_think_rate": sum(no_think(a) for a in answers) / n,
        "one_paragraph_rate": sum(one_paragraph(a) for a in answers) / n,
        "markdown_bullet_rate": sum(has_bullets(a) for a in answers) / n,
        "json_or_code_fence_rate": sum(has_code_or_json(a) for a in answers) / n,
        "empty_answer_rate": sum(not a.strip() for a in answers) / n,
        "answer_word_count_mean": sum(word_counts) / len(word_counts) if word_counts else None,
        "answer_word_count_p95": sorted(word_counts)[int(0.95 * (len(word_counts) - 1))] if word_counts else None,
        "latency_sec_mean": sum(float(r.get("latency_sec", 0) or 0) for r in pred_rows) / max(1, len(pred_rows)),
    }

ft_rule_metrics = compute_rule_metrics(ft_predictions)
openai_rule_metrics = compute_rule_metrics(openai_predictions)

print("FT rule metrics:", json.dumps(ft_rule_metrics, ensure_ascii=False, indent=2))
print("OpenAI rule metrics:", json.dumps(openai_rule_metrics, ensure_ascii=False, indent=2))

In [ ]:
# 7) Gemini round-robin pairwise judge

from google import genai
from google.genai import types

PAIRWISE_JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "ft": {
            "type": "object",
            "properties": {
                "correctness": {"type": "integer", "minimum": 1, "maximum": 5},
                "coverage": {"type": "integer", "minimum": 1, "maximum": 5},
                "groundedness": {"type": "integer", "minimum": 1, "maximum": 5},
                "clarity": {"type": "integer", "minimum": 1, "maximum": 5},
                "format": {"type": "integer", "minimum": 1, "maximum": 5},
                "hallucination": {"type": "boolean"}
            },
            "required": ["correctness", "coverage", "groundedness", "clarity", "format", "hallucination"]
        },
        "openai": {
            "type": "object",
            "properties": {
                "correctness": {"type": "integer", "minimum": 1, "maximum": 5},
                "coverage": {"type": "integer", "minimum": 1, "maximum": 5},
                "groundedness": {"type": "integer", "minimum": 1, "maximum": 5},
                "clarity": {"type": "integer", "minimum": 1, "maximum": 5},
                "format": {"type": "integer", "minimum": 1, "maximum": 5},
                "hallucination": {"type": "boolean"}
            },
            "required": ["correctness", "coverage", "groundedness", "clarity", "format", "hallucination"]
        },
        "winner": {"type": "string", "enum": ["ft", "openai", "tie"]},
        "reason": {"type": "string", "description": "Short Vietnamese explanation under 80 words."}
    },
    "required": ["ft", "openai", "winner", "reason"]
}

def get_thinking_level(name):
    name = str(name or "HIGH").upper()
    valid = {"MINIMAL", "LOW", "MEDIUM", "HIGH", "THINKING_LEVEL_UNSPECIFIED"}
    if name not in valid:
        raise ValueError(f"Unsupported thinking level: {name}")
    return getattr(types.ThinkingLevel, name)

def is_rate_limit_error(exc):
    code = getattr(exc, "code", None)
    msg = str(getattr(exc, "message", "") or exc).lower()
    return code == 429 or "rate limit" in msg or "quota" in msg or "resource exhausted" in msg or "high demand" in msg

def extract_json_obj(text):
    text = str(text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
    return None

class GeminiRoundRobin:
    def __init__(self, keys, model_name, rpm=15, rpd=500, cooldown=60):
        self.clients = [genai.Client(api_key=k) for k in keys]
        self.model_name = model_name
        self.rpm = rpm
        self.rpd = rpd
        self.cooldown = cooldown
        self.next_idx = 0
        self.calls = [0 for _ in keys]
        self.rate_limit_errors = [0 for _ in keys]
        self.rate_limited_until = [0.0 for _ in keys]
        self.last_call = [0.0 for _ in keys]
        self.usage = {"prompt_token_count": 0, "candidates_token_count": 0, "thoughts_token_count": 0, "total_token_count": 0}

    def _take_slot(self):
        now = time.monotonic()
        for _ in range(len(self.clients)):
            idx = self.next_idx
            self.next_idx = (self.next_idx + 1) % len(self.clients)
            if self.calls[idx] >= self.rpd:
                continue
            if self.rate_limited_until[idx] > now:
                continue
            min_gap = 60.0 / max(1, self.rpm)
            wait = max(0.0, min_gap - (now - self.last_call[idx]))
            if wait:
                time.sleep(wait)
            self.calls[idx] += 1
            self.last_call[idx] = time.monotonic()
            return idx, self.clients[idx]
        wait_until = min(self.rate_limited_until) if self.rate_limited_until else time.monotonic() + self.cooldown
        wait = max(5.0, wait_until - time.monotonic())
        print(f"All Gemini slots cooling down; sleep {wait:.1f}s")
        time.sleep(wait)
        return self._take_slot()

    def _mark_rate_limited(self, idx, exc):
        self.rate_limit_errors[idx] += 1
        self.rate_limited_until[idx] = time.monotonic() + self.cooldown
        print(f"Gemini slot {idx} rate/quota/high-demand; switching. Error: {str(exc)[:250]}")

    def _record_usage(self, response):
        usage = getattr(response, "usage_metadata", None)
        if usage is None:
            return
        for k in self.usage:
            self.usage[k] += int(getattr(usage, k, 0) or 0)

    def call_json(self, prompt, schema, max_retries=None):
        max_retries = max_retries or MAX_RETRIES_PER_SAMPLE
        last_exc = None
        for attempt in range(1, max_retries + 1):
            idx, client = self._take_slot()
            try:
                config_kwargs = dict(
                    temperature=0,
                    max_output_tokens=GEMINI_MAX_OUTPUT_TOKENS,
                    response_mime_type="application/json",
                    thinking_config=types.ThinkingConfig(
                        thinking_level=get_thinking_level(GEMINI_THINKING_LEVEL)
                    ),
                )
                try:
                    config_kwargs["response_schema"] = schema
                    config = types.GenerateContentConfig(**config_kwargs)
                except TypeError:
                    config_kwargs.pop("response_schema", None)
                    config = types.GenerateContentConfig(**config_kwargs)
                resp = client.models.generate_content(model=self.model_name, contents=prompt, config=config)
                self._record_usage(resp)
                text = getattr(resp, "text", None) or str(resp)
                obj = extract_json_obj(text)
                if isinstance(obj, dict):
                    return obj
                raise ValueError(f"Cannot parse JSON. Prefix={text[:500]!r}")
            except Exception as exc:
                last_exc = exc
                if is_rate_limit_error(exc):
                    self._mark_rate_limited(idx, exc)
                    continue
                print(f"Gemini judge error attempt={attempt}: {str(exc)[:300]}")
                time.sleep(min(2 * attempt, 20))
        raise RuntimeError(f"Gemini judge failed after {max_retries} attempts: {last_exc}")

gemini_judge = GeminiRoundRobin(
    GEMINI_API_KEYS,
    GEMINI_MODEL,
    rpm=GEMINI_RPM_LIMIT_PER_KEY,
    rpd=GEMINI_RPD_LIMIT_PER_KEY,
    cooldown=GEMINI_RATE_LIMIT_COOLDOWN_SEC,
)

def build_pairwise_judge_prompt(row, ft_answer, openai_answer):
    context = get_context(row, max_chars=7000)
    question = get_question(row)
    reference = get_ref_answer(row)
    return f"""Bạn là giám khảo nghiêm ngặt cho bài hỏi đáp giáo dục.

Nhiệm vụ: chấm hai câu trả lời theo cùng reference answer và context bài giảng.

Rubric:
- correctness: đúng ý với answer label/reference, thang 1-5.
- coverage: phủ đủ ý quan trọng trong reference, thang 1-5.
- groundedness: được hỗ trợ bởi context/transcript, không bịa ngoài context, thang 1-5.
- clarity: rõ ràng, dễ hiểu, thang 1-5.
- format: một đoạn ngắn gọn, không bullet, không markdown, không JSON/code fence, thang 1-5.
- hallucination: true nếu có thông tin không được context/reference hỗ trợ.

Chọn winner:
- "ft" nếu câu trả lời FT tốt hơn rõ.
- "openai" nếu OpenAI tốt hơn rõ.
- "tie" nếu ngang nhau hoặc khác biệt nhỏ.

Context:
{context}

Question:
{question}

Reference answer / label:
{reference}

FT answer:
{ft_answer}

OpenAI answer:
{openai_answer}

Chỉ trả JSON đúng schema. Không thêm markdown.
""".strip()

def candidate_score(d):
    if not isinstance(d, dict):
        return None
    score = (
        0.35 * float(d.get("correctness", 0) or 0) / 5
        + 0.20 * float(d.get("coverage", 0) or 0) / 5
        + 0.25 * float(d.get("groundedness", 0) or 0) / 5
        + 0.10 * float(d.get("clarity", 0) or 0) / 5
        + 0.10 * float(d.get("format", 0) or 0) / 5
    )
    if bool(d.get("hallucination", False)):
        score -= 0.15
    return max(0.0, min(1.0, score))

In [ ]:
# 8) Run Gemini pairwise judge with resume

def run_pairwise_judge():
    ft_map = {r.get("eval_id"): r for r in ft_predictions if r.get("status", "ok") == "ok"}
    openai_map = {r.get("eval_id"): r for r in openai_predictions if r.get("status", "ok") == "ok"}

    if FORCE_REJUDGE and JUDGE_PATH.exists():
        backup = JUDGE_PATH.with_suffix(JUDGE_PATH.suffix + f".backup_{int(time.time())}")
        JUDGE_PATH.rename(backup)
        print("Backed up old judge file:", backup)

    judged = [] if FORCE_REJUDGE else read_jsonl_safe(JUDGE_PATH)
    done_ids = {r.get("eval_id") for r in judged if r.get("judge_status") == "ok"}

    print("Existing judged OK:", len(done_ids), "/", len(eval_rows))
    print("Judge output:", JUDGE_PATH)

    for row in tqdm(eval_rows, desc="Gemini pairwise judge", unit="sample"):
        eid = get_id(row)
        if eid in done_ids:
            continue
        if eid not in ft_map or eid not in openai_map:
            out = {"eval_id": eid, "split": row.get("split"), "judge_status": "missing_prediction", "has_ft": eid in ft_map, "has_openai": eid in openai_map}
            append_jsonl(JUDGE_PATH, out)
            judged.append(out)
            continue

        ft_answer = prediction_answer(ft_map[eid])
        openai_answer = prediction_answer(openai_map[eid])
        prompt = build_pairwise_judge_prompt(row, ft_answer, openai_answer)

        try:
            obj = gemini_judge.call_json(prompt, PAIRWISE_JUDGE_SCHEMA)
            ft_score = candidate_score(obj.get("ft"))
            openai_score = candidate_score(obj.get("openai"))
            out = {
                "eval_id": eid,
                "split": row.get("split"),
                "lecture_id": (row.get("metadata") or {}).get("lecture_id"),
                "difficulty": (row.get("metadata") or {}).get("difficulty"),
                "question_intent": (row.get("metadata") or {}).get("question_intent"),
                "judge_status": "ok",
                "winner": obj.get("winner"),
                "reason": obj.get("reason"),
                "ft_score": ft_score,
                "openai_score": openai_score,
                "score_delta_ft_minus_openai": None if ft_score is None or openai_score is None else ft_score - openai_score,
                "ft": obj.get("ft"),
                "openai": obj.get("openai"),
            }
        except Exception as exc:
            out = {"eval_id": eid, "split": row.get("split"), "judge_status": "failed", "judge_error": str(exc)[:1000]}
            print("[WARN] judge failed:", eid, exc)

        append_jsonl(JUDGE_PATH, out)
        judged.append(out)
        if out.get("judge_status") == "ok":
            done_ids.add(eid)

    return read_jsonl_safe(JUDGE_PATH)

if RUN_GEMINI_JUDGE:
    judged_rows = run_pairwise_judge()
else:
    judged_rows = read_jsonl_safe(JUDGE_PATH)

print("Judged rows:", len(judged_rows))
print("Gemini calls:", gemini_judge.calls)
print("Gemini usage:", json.dumps(gemini_judge.usage, ensure_ascii=False, indent=2))

In [ ]:
# 9) Aggregate results

def ok_judges(rows):
    return [r for r in rows if r.get("judge_status") == "ok"]

def aggregate_candidate_metrics(judged, key):
    rows = ok_judges(judged)
    dlist = [r.get(key, {}) for r in rows]
    return {
        "sample_count": len(rows),
        "score_mean": mean([r.get(f"{key}_score") for r in rows]),
        "correctness_mean": mean([float(d.get("correctness")) for d in dlist if d.get("correctness") is not None]),
        "coverage_mean": mean([float(d.get("coverage")) for d in dlist if d.get("coverage") is not None]),
        "groundedness_mean": mean([float(d.get("groundedness")) for d in dlist if d.get("groundedness") is not None]),
        "clarity_mean": mean([float(d.get("clarity")) for d in dlist if d.get("clarity") is not None]),
        "format_mean": mean([float(d.get("format")) for d in dlist if d.get("format") is not None]),
        "hallucination_rate": mean([1.0 if d.get("hallucination") else 0.0 for d in dlist if "hallucination" in d]),
    }

def aggregate_win_rate(judged):
    rows = ok_judges(judged)
    c = Counter(r.get("winner", "unknown") for r in rows)
    n = len(rows) or 1
    return {
        "compared": len(rows),
        "ft_win": c.get("ft", 0),
        "openai_win": c.get("openai", 0),
        "tie": c.get("tie", 0),
        "ft_win_rate": c.get("ft", 0) / n,
        "openai_win_rate": c.get("openai", 0) / n,
        "tie_rate": c.get("tie", 0) / n,
    }

def aggregate_by_split(judged):
    out = {}
    for split in sorted(set(r.get("split", "unknown") for r in judged)):
        sub = [r for r in judged if r.get("split", "unknown") == split]
        out[split] = {
            "win_rate": aggregate_win_rate(sub),
            "ft": aggregate_candidate_metrics(sub, "ft"),
            "openai": aggregate_candidate_metrics(sub, "openai"),
        }
    return out

ft_metrics = aggregate_candidate_metrics(judged_rows, "ft")
openai_metrics = aggregate_candidate_metrics(judged_rows, "openai")
win_rate = aggregate_win_rate(judged_rows)
by_split = aggregate_by_split(judged_rows)

summary = {
    "config": {
        "dataset_path": str(DATA_JSONL_PATH),
        "base_model": BASE_MODEL,
        "checkpoint_path": str(CHECKPOINT_PATH),
        "fallback_adapter_path": str(FINAL_ADAPTER_PATH),
        "openai_predict_model": OPENAI_PREDICT_MODEL,
        "gemini_judge_model": GEMINI_MODEL,
        "gemini_thinking_level": GEMINI_THINKING_LEVEL,
        "max_eval_samples": MAX_EVAL_SAMPLES,
        "gemini_calls": gemini_judge.calls,
        "gemini_usage": gemini_judge.usage,
    },
    "paths": {
        "ft_predictions": str(FT_PRED_PATH),
        "openai_predictions": str(OPENAI_PRED_PATH),
        "judge": str(JUDGE_PATH),
        "summary": str(SUMMARY_OUT),
        "comparison_csv": str(COMPARISON_CSV_OUT),
        "pairwise_csv": str(PAIRWISE_CSV_OUT),
    },
    "rule_metrics": {"ft": ft_rule_metrics, "openai": openai_rule_metrics},
    "judge_metrics": {"ft": ft_metrics, "openai": openai_metrics},
    "win_rate": win_rate,
    "by_split": by_split,
}

write_json(SUMMARY_OUT, summary)

comparison_rows = []
for group, a, b in [("rule", ft_rule_metrics, openai_rule_metrics), ("judge", ft_metrics, openai_metrics)]:
    for k in sorted(set(a) & set(b)):
        if isinstance(a[k], (int, float)) and isinstance(b[k], (int, float)):
            comparison_rows.append({"group": group, "metric": k, "ft": a[k], "openai": b[k], "delta_ft_minus_openai": a[k] - b[k]})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(COMPARISON_CSV_OUT, index=False)

pairwise_rows = []
for r in ok_judges(judged_rows):
    pairwise_rows.append({
        "eval_id": r.get("eval_id"),
        "split": r.get("split"),
        "lecture_id": r.get("lecture_id"),
        "difficulty": r.get("difficulty"),
        "question_intent": r.get("question_intent"),
        "winner": r.get("winner"),
        "ft_score": r.get("ft_score"),
        "openai_score": r.get("openai_score"),
        "delta_ft_minus_openai": r.get("score_delta_ft_minus_openai"),
        "reason": r.get("reason"),
        "ft_correctness": (r.get("ft") or {}).get("correctness"),
        "openai_correctness": (r.get("openai") or {}).get("correctness"),
        "ft_groundedness": (r.get("ft") or {}).get("groundedness"),
        "openai_groundedness": (r.get("openai") or {}).get("groundedness"),
        "ft_hallucination": (r.get("ft") or {}).get("hallucination"),
        "openai_hallucination": (r.get("openai") or {}).get("hallucination"),
    })

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv(PAIRWISE_CSV_OUT, index=False)

print(json.dumps(summary, ensure_ascii=False, indent=2)[:6000])
print("Saved:", SUMMARY_OUT)
display(comparison_df)
display(pairwise_df.head(20))

In [ ]:
# 10) Random review examples

if RUN_RANDOM_REVIEW:
    ft_map = {r.get("eval_id"): r for r in ft_predictions}
    openai_map = {r.get("eval_id"): r for r in openai_predictions}
    row_map = {get_id(r): r for r in eval_rows}
    ok_rows = ok_judges(judged_rows)

    rng = random.Random(EVAL_SEED)
    sample = rng.sample(ok_rows, k=min(8, len(ok_rows)))

    review_rows = []
    for j in sample:
        eid = j.get("eval_id")
        src = row_map[eid]
        review_rows.append({
            "eval_id": eid,
            "split": j.get("split"),
            "question": get_question(src),
            "reference_answer": get_ref_answer(src),
            "ft_answer": prediction_answer(ft_map[eid]),
            "openai_answer": prediction_answer(openai_map[eid]),
            "winner": j.get("winner"),
            "ft_score": j.get("ft_score"),
            "openai_score": j.get("openai_score"),
            "reason": j.get("reason"),
            "ft_judge": j.get("ft"),
            "openai_judge": j.get("openai"),
        })

    review_df = pd.DataFrame(review_rows)
    overwrite_jsonl(RANDOM_REVIEW_JSONL, review_rows)
    review_df.to_csv(RANDOM_REVIEW_CSV, index=False)

    md = []
    for r in review_rows:
        md.append(f"## {r['eval_id']} | winner={r['winner']}\n")
        md.append(f"**Câu hỏi**\n\n{r['question']}\n")
        md.append(f"**Answer label/reference**\n\n{r['reference_answer']}\n")
        md.append(f"**FT answer**\n\n{r['ft_answer']}\n")
        md.append(f"**OpenAI answer**\n\n{r['openai_answer']}\n")
        md.append(f"**Điểm**\n\nFT={r['ft_score']} | OpenAI={r['openai_score']}\n\n")
        md.append(f"**Lý do judge**\n\n{r['reason']}\n")
        md.append("\n---\n")

    RANDOM_REVIEW_MD.write_text("\n".join(md), encoding="utf-8")
    display(review_df)
    print("Saved random review:", RANDOM_REVIEW_MD)

In [ ]:
# 11) Quick status
print("Eval dir:", EVAL_DIR)
print("FT predictions:", FT_PRED_PATH, FT_PRED_PATH.exists())
print("OpenAI predictions:", OPENAI_PRED_PATH, OPENAI_PRED_PATH.exists())
print("Judge:", JUDGE_PATH, JUDGE_PATH.exists())
print("Summary:", SUMMARY_OUT, SUMMARY_OUT.exists())